In [ ]:

# 🧪 Cell 2: Environment Setup
from dotenv import load_dotenv
import os

# Load from .env if available
load_dotenv()

# Set Langfuse credentials explicitly (for testing, avoid hardcoding in production)
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-33d19971-9554-435a-88b6-b85d5e4697cb"
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-212bbb34-e639-4653-acd6-ac37030756f5"
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"


# 🧪 Cell 3: Import Langfuse and Setup Callback Handler

from langfuse import Langfuse  # ✅ this is the class


from langfuse.langchain.CallbackHandler import LangchainCallbackHandler as LangfuseCallbackHandler

langfuse_handler = LangfuseCallbackHandler()


#print("Langfuse SDK Version:", langfuse_client._client.config.sdk_version)


# 🧪 Cell 4: Setup Ollama LLM (or OpenAI if needed)
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="llama3",
    temperature=0.7,
    callbacks=[langfuse_handler]
)


# 🧪 Cell 5: Define Tools (Search example)
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import Tool

search_tool = DuckDuckGoSearchRun()

tools = [
    Tool(
        name="Search",
        func=search_tool.run,
        description="Useful for answering questions about current events or factual info"
    )
]


# 🧪 Cell 6: Create ReAct Agent and Executor
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

react_prompt = PromptTemplate.from_template("""Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: what you should do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat 2 times)
Thought: I now know the final answer
Final Answer: the final answer to the original question

Begin!

Question: {input}
{agent_scratchpad}""")

agent = create_react_agent(llm=llm, tools=tools, prompt=react_prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    callbacks=[langfuse_handler],
    max_iterations=3,           # default is 15
    max_execution_time=60       # in seconds
)


# 🧪 Cell 8: Run the Agent with Full Langfuse Trace
question = "When is the best time to visit Switzerland?"

#with langfuse_client.trace(name="sankari-query-trace") as trace:
 #   with trace.span(name="agent-execution"):
response = agent_executor.invoke(
            {"input": question},
            config={"callbacks": [langfuse_handler], "run_name": "sankari-NEW-query"}
        )

print("\n✅ Final Answer:")
print(response["output"])

